In [121]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from pydantic import Field, BaseModel
import operator

In [122]:
load_dotenv()

model = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3,
)

In [123]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="Sentiment of the review"
    )

In [124]:
structured_model = model.with_structured_output(SentimentSchema)

In [153]:
prompt = """ 

The hospital infrastructure is good, but the management system needs improvement. Appointments were delayed, and the staff seemed unorganized. The billing process was confusing.
"""

In [126]:
class review_state(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    response:str
    diagnosis:str

In [127]:
def find_sentiment(state:review_state):
    sentiment=structured_model.invoke(state['review'])
    return {'sentiment':sentiment}
    

In [140]:
def check_sentiment(state:review_state)->Literal["positive_response","generate_diagnosis"]:
    if state['sentiment'].sentiment=='positive':
        return "positive_response"
    else:
        return "generate_diagnosis" 

        


In [141]:
def positive_response(state:review_state):
    prompt=f"you need to generate positive response for the customer according to review:{state['review']}"
    response=model.invoke(prompt)
    return {'response':response}

In [142]:
def generate_diagnosis(state:review_state):
    prompt=f"you need to generate issue , problem according to reivew:{state['review']} and sentiment:{state['sentiment']}"
    diagnosis=model.invoke(prompt)


    return {'diagnosis':diagnosis}

In [143]:
def negative_response(state:review_state):
    prompt=f"you need to generate negative response for the customer according to review:{state['review']}"
    response=model.invoke(prompt)
    return {'response':response}



In [144]:
# define graph
graph = StateGraph(review_state)

#define node
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('generate_diagnosis',generate_diagnosis)
graph.add_node('negative_response',negative_response)

#define edge
graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('generate_diagnosis','negative_response')
graph.add_edge('positive_response',END)
graph.add_edge('negative_response',END)


#compile

workflow=graph.compile()


In [154]:
initial_state={
    'review':prompt
}

final_state=workflow.invoke(initial_state)

In [155]:
final_state['sentiment'].sentiment

'negative'

In [156]:
final_state['response'].content

"Thank you for taking the time to share your feedback with us. We sincerely apologize for the frustration and inconvenience you experienced during your recent visit.\n\nWe are glad you noted our hospital's infrastructure, but we are truly sorry to hear that our management systems, appointment scheduling, staff coordination, and billing process did not meet your expectations. Your experience with delays, disorganization, and confusing billing is not the standard we strive for, and we take your concerns very seriously.\n\nWe are already reviewing your specific points with our operations and administrative teams to identify the root causes. Immediate steps are being initiated to:\n1.  **Improve appointment scheduling efficiency** to reduce wait times.\n2.  **Enhance staff training and communication protocols** to ensure better coordination and a more organized experience.\n3.  **Simplify and clarify our billing statements and explanations** to make the process more transparent and underst

In [158]:
final_state['diagnosis'].content


'**Issue Title:** Systemic Management Failures Undermining Patient Experience Despite Adequate Physical Infrastructure\n\n**Problem Statement:**\nWhile the hospital\'s physical infrastructure and facilities meet acceptable standards, critical failures in the management and administrative systems are creating a significantly negative and frustrating patient experience. These operational inefficiencies directly contradict the quality implied by the hospital\'s physical assets.\n\n**Specific Pain Points:**\n1.  **Appointment Scheduling & Flow:** The appointment system is dysfunctional, leading to excessive and unexplained delays. This indicates poor time management, inadequate patient volume planning, and a lack of real-time communication about wait times.\n2.  **Staff Coordination & Organization:** Front-line and clinical staff appear disorganized and lack clear protocols. This manifests as poor patient guidance, inconsistent information, and a breakdown in inter-departmental communicati

In [159]:
print(final_state)

{'review': ' \n\nThe hospital infrastructure is good, but the management system needs improvement. Appointments were delayed, and the staff seemed unorganized. The billing process was confusing.\n', 'sentiment': SentimentSchema(sentiment='negative'), 'response': AIMessage(content="Thank you for taking the time to share your feedback with us. We sincerely apologize for the frustration and inconvenience you experienced during your recent visit.\n\nWe are glad you noted our hospital's infrastructure, but we are truly sorry to hear that our management systems, appointment scheduling, staff coordination, and billing process did not meet your expectations. Your experience with delays, disorganization, and confusing billing is not the standard we strive for, and we take your concerns very seriously.\n\nWe are already reviewing your specific points with our operations and administrative teams to identify the root causes. Immediate steps are being initiated to:\n1.  **Improve appointment schedu